Step 1: 讀資料
  df = pd.read_csv('data.csv')

Step 2: 探索資料
  df.head()
  df.isnull().sum()
  df.describe()

Step 3: 特徵創造（Pipeline 前）
  df['新特徵'] = df['A'] / df['B']
  pd.cut() 分群

Step 4: 建立 Pipeline
  SimpleImputer → StandardScaler → Model

Step 5: K-Fold 交叉驗證
  每個 fold 都用 Pipeline
  避免資料洩漏

Step 6: 超參數調整
  Optuna 找最佳參數

Step 7: 評估
  Accuracy, F1, AUC

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import  accuracy_score, f1_score





# df.loc[np.random.choice(df.index, 20), '年齡'] = np.nan
.loc[列, 欄] → 指定那些列的「年齡」  
np.random.choice(df.index, 20) → 隨機挑 20 列  
df.index → 所有列的索引  
設成 np.nan → 模擬缺失值  


In [23]:
np.random.seed(42)
n = 500

data = {
    '年齡': np.random.randint(18, 65, n),
    '收入': np.random.randint(20000, 200000, n),
    '消費金額': np.random.randint(1000, 50000, n),
    '購買次數': np.random.randint(1, 50, n),
    '上次購買天數': np.random.randint(1, 365, n),
}

df = pd.DataFrame(data)
df.loc[np.random.choice(df.index, 20), '年齡'] = np.nan
df.loc[np.random.choice(df.index, 15), '收入'] = np.nan

df['再購買'] = ((df['購買次數'] > 25) & (df['上次購買天數'] < 180)).astype(int)

# 特徵創造（Pipeline 前先做）
df['平均消費'] = df['消費金額'] / df['購買次數']
df['消費收入比'] = df['消費金額'] / df['收入']
df['年齡層'] = pd.cut(df['年齡'].fillna(df['年齡'].median()), 
                      bins=[0, 30, 45, 65], 
                      labels=[0, 1, 2]).astype(int)

print(f"資料形狀: {df.shape}")
print(f"新增特徵後的欄位數: {len(df.columns)}")

資料形狀: (500, 9)
新增特徵後的欄位數: 9


# pipeline = Pipeline([  
    ('imputer', SimpleImputer(strategy='mean')),        #用「平均值」補 NaN  
    ('scaler', StandardScaler()),                       #讓資料標準化  
    ('model', RandomForestClassifier(random_state=42))  #模型RandomForestClassifier(random_state=42)  
])  
 
# for name, step in pipeline.steps:
把 pipeline 裡每個步驟，一個一個拿出來
column 0 叫name，column 1 叫step 
# print(f"  {name}: {step.__class__.__name__}")  
{name}   
    印出你取的名字  
step.class.name   
    問 Python：「這個物件是什麼類別？」  

In [24]:
# 分離特徵和目標
X = df.drop('再購買', axis=1)
y = df['再購買']

# 建立 Pipeline
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state=42))
])

print("Pipeline 步驟:")
for name, step in pipeline.steps:
    print(f"  {name}: {step.__class__.__name__}")

Pipeline 步驟:
  imputer: SimpleImputer
  scaler: StandardScaler
  model: RandomForestClassifier


# shuffle=True  
在切之前：先把資料「打亂」 

# for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
KFold 幫你切 5 次資料，現在開始一輪一輪跑
kf.split(X)   
    每一輪會回傳：  
    train_idx = 訓練資料索引  
    val_idx = 驗證資料索引  
enumerate 的作用  
    fold = 0,1,2,3,4  
# X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]...
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

用 index 去切 DataFrame  

名稱	用途  
X_train	拿來學習  
X_val	拿來考試  
y_train	正確答案（訓練）  
y_val	正確答案（測試）
# scores_acc.append(acc)
scores_f1.append(f1)  

👉 每一折的結果都存起來   
# np.std（標準差）
numpy  

In [25]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores_acc = []
scores_f1 = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Pipeline 自動做：缺失值 → 標準化 → 訓練
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    
    scores_acc.append(acc)
    scores_f1.append(f1)
    
    print(f"Fold {fold+1}: Accuracy={acc:.4f}, F1={f1:.4f}")

print(f"\n平均 Accuracy: {np.mean(scores_acc):.4f} ± {np.std(scores_acc):.4f}")
print(f"平均 F1:       {np.mean(scores_f1):.4f} ± {np.std(scores_f1):.4f}")

Fold 1: Accuracy=0.9900, F1=0.9756
Fold 2: Accuracy=1.0000, F1=1.0000
Fold 3: Accuracy=0.9900, F1=0.9804
Fold 4: Accuracy=1.0000, F1=1.0000
Fold 5: Accuracy=1.0000, F1=1.0000

平均 Accuracy: 0.9960 ± 0.0049
平均 F1:       0.9912 ± 0.0109


Optuna
   ↓
試參數（trial）
   ↓
KFold 評估
   ↓
Pipeline 訓練
   ↓
回傳 F1
   ↓
找最大值

In [26]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
    }
    
    pipeline_opt = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(**params, random_state=42))
    ])
    
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        pipeline_opt.fit(X_train, y_train)
        score = f1_score(y_val, pipeline_opt.predict(X_val))
        scores.append(score)
    
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print(f"最佳參數: {study.best_params}")
print(f"最佳 F1: {study.best_value:.4f}")

最佳參數: {'n_estimators': 126, 'max_depth': 7}
最佳 F1: 0.9912
